In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ["NO_ALBUMENTATIONS_UPDATE"] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
import keras
print(keras.__version__)

In [ ]:
import logging
logger = tf.get_logger()
logger.setLevel(logging.ERROR)

In [ ]:
import cv2
import math
import time
import shutil 
import random
import itertools
import numpy as np
import pandas as pd 
from glob import glob
import albumentations as A
from tensorflow import keras
from functools import partial
import matplotlib.pyplot as plt
from tensorflow.keras import backend as K

In [ ]:
# tf.keras.saving.get_custom_objects().clear()
# tf.keras.backend.clear_session() ## to clear other sessions in the environment
tf.config.optimizer.set_jit(False) ## enabling XLA

In [ ]:
# tf.keras.mixed_precision.set_global_policy(tf.keras.mixed_precision.Policy('mixed_float16'))

In [ ]:
tf.config.list_physical_devices('GPU')

# Variables

In [ ]:
EPOCHS=200
IMG_SIZE=224
BATCH_SIZE=512
LEARNING_RATE=0.0001
dropout_rate=0.1
BUFFER_SIZE=1000
AUTO=tf.data.AUTOTUNE
EARLY_STOPPING_PATIENCE=20

In [ ]:
tempfolder="temp_ISIC2019"
isExist = os.path.exists(tempfolder)
if not isExist:
    os.makedirs(tempfolder)

In [ ]:
end = '_HAM10000_Siamese'
destination='Results/siamese/2.ISIC2019/'

train_csv_file="fold1_train.csv" 
test_csv_file="fold1_test.csv" 

output_filename_train_genuine = "ISIC2019_siamese_train_genuine_pairs.csv"
output_filename_train_imposter = "ISIC2019_siamese_train_imposter_pairs.csv"  

output_filename_test_genuine = "ISIC2019_siamese_test_genuine_pairs.csv"
output_filename_test_imposter = "ISIC2019_siamese_test_imposter_pairs.csv"  

data_root="Datasets/2.ISIC2019/rasm/siamese/fold1/"
path="Datasets/2.ISIC2019/"
output_path="./"+tempfolder+"/"

In [ ]:
num_cl=len(os.listdir(data_root+"train/"))
print(num_cl)

# Load Data

In [ ]:
genuine_train=np.array(pd.read_csv(data_root+output_filename_train_genuine, names=["0","1","2"], header=None))

genuine_pairs_train=[]
genuine_labels_train=[]
for i in range(genuine_train.shape[0]):
    image1=genuine_train[i][0]
    image2=genuine_train[i][1]
    label=genuine_train[i][2]    
    genuine_pairs_train.append([image1,image2])
    genuine_labels_train.append(label)

genuine_pairs_train=np.array(genuine_pairs_train)
genuine_labels_train=np.array(genuine_labels_train)

In [ ]:
print(genuine_pairs_train.shape, genuine_labels_train.shape)

In [ ]:
imposter_train=np.array(pd.read_csv(data_root+output_filename_train_imposter, names=["0","1","2"], header=None))
imposter_pairs_train=[]
imposter_labels_train=[]
for i in range(imposter_train.shape[0]):
    image1=imposter_train[i][0]
    image2=imposter_train[i][1]
    label=imposter_train[i][2]    
    imposter_pairs_train.append([image1,image2])
    imposter_labels_train.append(label)

imposter_pairs_train=np.array(imposter_pairs_train)
imposter_labels_train=np.array(imposter_labels_train)

In [ ]:
print(imposter_pairs_train.shape, imposter_labels_train.shape)

In [ ]:
pairs_train=np.concatenate((genuine_pairs_train, imposter_pairs_train), axis=0)
labels_train=np.concatenate((genuine_labels_train, imposter_labels_train), axis=0)

In [ ]:
print(pairs_train.shape, labels_train.shape)

In [ ]:
genuine_test=np.array(pd.read_csv(data_root+output_filename_test_genuine, names=["0","1","2"], header=None))
genuine_pairs_test=[]
genuine_labels_test=[]
for i in range(genuine_test.shape[0]):
    image1=genuine_test[i][0]
    image2=genuine_test[i][1]
    label=genuine_test[i][2]    
    genuine_pairs_test.append([image1,image2])
    genuine_labels_test.append(label)

genuine_pairs_test=np.array(genuine_pairs_test)
genuine_labels_test=np.array(genuine_labels_test)

In [ ]:
print(genuine_pairs_test.shape, genuine_labels_test.shape)

In [ ]:
imposter_test=np.array(pd.read_csv(data_root+output_filename_test_imposter, names=["0","1","2"], header=None))
imposter_pairs_test=[]
imposter_labels_test=[]
for i in range(imposter_test.shape[0]):
    image1=imposter_test[i][0]
    image2=imposter_test[i][1]
    label=imposter_test[i][2]    
    imposter_pairs_test.append([image1,image2])
    imposter_labels_test.append(label)

imposter_pairs_test=np.array(imposter_pairs_test)
imposter_labels_test=np.array(imposter_labels_test)

In [ ]:
print(imposter_pairs_test.shape, imposter_labels_test.shape)

In [ ]:
pairs_test=np.concatenate((genuine_pairs_test, imposter_pairs_test), axis=0)
labels_test=np.concatenate((genuine_labels_test, imposter_labels_test), axis=0)

In [ ]:
print(pairs_test.shape, labels_test.shape)

In [ ]:
print(pairs_train.shape, labels_train.shape, pairs_test.shape, labels_test.shape)

In [ ]:
def preprocess_image(file_path, label):
    def load_and_preprocess(file_path):        
        image = tf.io.read_file(path + file_path)
        image = tf.image.decode_jpeg(image, channels=3, dct_method="INTEGER_ACCURATE")
        image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method='area')
        return image

    image_a = load_and_preprocess(file_path[0])
    image_b = load_and_preprocess(file_path[1])
    
    return (image_a, image_b), label

In [ ]:
def create_tf_train_dataset(file_paths, labels, batch_size):
    dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    dataset = dataset.shuffle(buffer_size=BUFFER_SIZE)
    dataset = dataset.map(lambda x, y: preprocess_image(x, y), num_parallel_calls=AUTO)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=AUTO)
    return dataset

train_ds=create_tf_train_dataset(pairs_train, labels_train, BATCH_SIZE)

In [ ]:
def create_tf_test_dataset(file_paths, labels, batch_size):
    dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    dataset = dataset.shuffle(buffer_size=BUFFER_SIZE)
    dataset = dataset.map(lambda x, y: preprocess_image(x, y), num_parallel_calls=AUTO)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=AUTO)
    return dataset

test_ds=create_tf_test_dataset(pairs_test, labels_test, BATCH_SIZE)

In [ ]:
def convert(seconds):
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60     
    return "%d:%02d:%02d" % (hour, minutes, seconds)

In [ ]:
ModelCheckPoint=tf.keras.callbacks.ModelCheckpoint(
    filepath=output_path+'EfficientNetB0-{epoch:03d}-{val_accuracy:.4f}.weights.h5',
    monitor='val_accuracy', verbose=1, save_best_only=True, save_weights_only=True)

EarlyStop=tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True, min_delta=0.001, verbose=1)

# Class Weights

In [ ]:
tr_lb=np.array(labels_train)
tr_lb_unique=np.unique(tr_lb)
print((np.unique(tr_lb, return_counts=True)))

In [ ]:
from sklearn.utils import compute_class_weight
#formula: n_samples / (n_classes * np.bincount(y))
classWeight=compute_class_weight(class_weight="balanced",classes=tr_lb_unique,y=tr_lb)
classWeight=np.round(classWeight,4)
classWeight_dict = dict(zip(tr_lb_unique, classWeight))
print("classWeight : ",classWeight_dict)

# Prepare Model

In [ ]:
inputs=tf.keras.layers.Input(shape=(IMG_SIZE,IMG_SIZE,3))
base_model=tf.keras.applications.EfficientNetB0(include_top=False, input_tensor=inputs, weights='imagenet')
base_model.trainable=False

till=base_model.layers.index(base_model.get_layer('block5c_add'))+1
for layer in base_model.layers[till:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

x = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(base_model.output)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(dropout_rate, name="dropout1")(x)
x = tf.keras.layers.Dense(512, activation="silu", name="pred1")(x) 
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(dropout_rate, name="dropout2")(x)
x = tf.keras.layers.Dense(128, activation="silu", name="pred2")(x) 
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(dropout_rate, name="dropout3")(x)
outputs = tf.keras.layers.Dense(32, activation="softmax", name="pred3")(x) 
embedding_network=tf.keras.Model(inputs, outputs, name="Embedding_EfficientNetB0")
embedding_network.load_weights("EfficientNetB0-086-0.9670.weights.h5", by_name=True, skip_mismatch=True)

In [ ]:
@tf.keras.utils.register_keras_serializable(package='Custom', name='SquaredEuclideanDistance')
class SquaredEuclideanDistance(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        anchor, positive = inputs
        squared_diff = tf.square(anchor - positive)
        return tf.reduce_sum(squared_diff, axis=-1, keepdims=True)

    def get_config(self):
        config = super().get_config()
        return config

In [ ]:
input_1 = tf.keras.Input(shape=(IMG_SIZE,IMG_SIZE,3))
input_2 = tf.keras.Input(shape=(IMG_SIZE,IMG_SIZE,3))

tower_1 = embedding_network(input_1)
tower_2 = embedding_network(input_2)

merge_layer=SquaredEuclideanDistance()([tower_1, tower_2])

siamese=tf.keras.Model(inputs=[input_1, input_2], outputs=merge_layer, name="Siamese")

In [ ]:
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=LEARNING_RATE, decay_steps=500, decay_rate=0.9
)

In [ ]:
siamese.compile(
    loss="binary_crossentropy", 
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule), 
    metrics=["accuracy"]
)

In [ ]:
t1=time.time()
history=siamese.fit(
    train_ds, 
    validation_data=test_ds,  
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    class_weight=classWeight_dict,
    callbacks=[ModelCheckPoint, EarlyStop] 
)
t2=time.time()
time_taken = t2-t1
print("Time taken : ",convert(time_taken)," hh:mm:ss")

In [ ]:
Hist = {'hist' : history.history} 
df = pd.DataFrame.from_dict(Hist)  
df.to_csv(output_path+'Hist.csv', index = True, header=True)  

# Save Model

In [ ]:
f=output_path+'EfficientNetB0-009-0.9297.weights.h5'
siamese.load_weights(f) 

In [ ]:
test_loss, test_accuracy_by_evaluate = siamese.evaluate(test_ds)

In [ ]:
test_accuracy_by_evaluate_rounded = round((test_accuracy_by_evaluate)*100, 2)

In [ ]:
num_epoch = len(history.history['loss'])
executed_epoch_count = len(history.history['loss'])

In [ ]:
savedfilename="epochs_{}".format(executed_epoch_count)+"_test_acc_{}".format(test_accuracy_by_evaluate_rounded)

In [ ]:
name=output_path+"{}".format(savedfilename)+".keras"
siamese.save(name)

In [ ]:
name_weights=output_path+"{}".format(savedfilename)+".weights.h5"
siamese.save_weights(name_weights)

In [ ]:
figpath=output_path+"{}".format(savedfilename)+".pdf"
fig = plt.figure(figsize=(10,4))
epochs_range = range(len(history.history['loss']))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history['accuracy'], label='Train')
plt.plot(epochs_range, history.history['val_accuracy'], label='Val')
plt.legend(loc='lower right')
plt.title('Accuracy')
plt.xlabel('epochs')
plt.ylabel('accuracy')
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history['loss'], label='Train')
plt.plot(epochs_range, history.history['val_loss'], label='Val')
plt.legend(loc='upper right')
plt.title('Loss')
plt.xlabel('epochs')
plt.ylabel('loss')
plt.show()
fig.savefig(figpath, format='pdf', bbox_inches='tight')

# Predictions

In [ ]:
for x in range(10):
    print(pairs_test[x][0], pairs_test[x][1], labels_test[x])

In [ ]:
test_pairs=[]
l=sorted(glob(data_root+"test_aug/*/*"))
for x in l:
    path=x
    lis=sorted(os.listdir(str(x)))
    for y in lis:
        d=x.split('/')[-1]
        f1=x+'/'+d+'_Original.png'
        f2=x+'/'+y
        test_pairs.append([f1,f2])

test_pairs=np.array(test_pairs)
df=pd.DataFrame(test_pairs)
df.to_csv(output_path+'test_pairs.csv', header=None, index=False)

In [ ]:
pr=[]
pa=np.array(pd.read_csv(output_path+'test_pairs.csv', header=None))
for x in pa:
    im1=cv2.imread(x[0])
    im1=cv2.resize(im1, (IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_AREA)
    im1=np.expand_dims(im1, axis=0)
    im2=cv2.imread(x[1])
    im2=cv2.resize(im2, (IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_AREA)
    im2=np.expand_dims(im2, axis=0)
    pred=siamese.predict([im1,im2], verbose=0)
    pred=np.round(pred[0][0], 4)
    pr.append([x[0], x[1], pred])

pr=np.array(pr) 
df=pd.DataFrame(pr)
df.to_csv(output_path+'test_predictions.csv', header=None, index=False)

In [ ]:
transform_list = ['AutoContrast', 'BoxBlur', 'CLAHE', 'ChannelShuffle', 'ChromaticAberration', 'CoarseDropout', 
                  'ColorJitter', 'Defocus', 'Dilation', 'ElasticTransform', 'Emboss', 'Equalize', 'EqualizeYUV', 
                  'Erosion', 'Frost', 'Gamma', 'GaussNoise', 'GaussianBlur', 'GlassBlur', 'HorizontalFlip', 
                  'HueSaturationValue', 'Invert', 'JPEGCompression', 'MedianBlur', 'MotionBlur', 'OpticalDistortion', 
                  'Original', 'Perspective', 'Posterize', 'RGBShift', 'RandomBrightnessContrast', 'RandomFog', 
                  'RandomGravel', 'RandomGridShuffle', 'RandomRain', 'RandomShadow', 'RandomSnow', 'RandomSunFlare', 
                  'RandomToneCurve', 'Rotate', 'SaltandPepper', 'Scale', 'Sharpen', 'ShearX', 'ShearY', 'ShiftScaleRotate', 
                  'Solarize', 'Speckle', 'ToGray', 'ToSepia', 'Translate', 'VerticalFlip', 'Vignette', 'ZoomBlur']

In [ ]:
# transform_list = ['AutoContrast', 'BoxBlur', 'CLAHE', 'ChannelShuffle', 'ChromaticAberration', 'CoarseDropout', 
#                   'ColorJitter', 'Defocus', 'Dilation', 'ElasticTransform', 'Emboss', 'Equalize', 'EqualizeYUV', 
#                   'Erosion', 'Frost', 'Gamma', 'GaussNoise', 'GaussianBlur', 'GlassBlur', 'HorizontalFlip', 
#                   'HueSaturationValue', 'Invert', 'JPEGCompression', 'MedianBlur', 'MotionBlur', 'OpticalDistortion', 
#                   'Original', 'Perspective', 'Posterize', 'RGBShift', 'RandomBrightnessContrast', 'RandomFog', 
#                   'RandomGravel', 'RandomGridShuffle', 'RandomRain', 'RandomShadow', 'RandomSnow', 'RandomSunFlare', 
#                   'RandomToneCurve', 'Rotate', 'SaltandPepper', 'Scale', 'Sharpen', 'ShearX', 'ShearY', 'ShiftScaleRotate', 
#                   'Solarize', 'Speckle', 'ToGray', 'ToSepia', 'Translate', 'VerticalFlip', 'Vignette', 'ZoomBlur']

In [ ]:
dt=np.array(pd.read_csv(output_path+'test_predictions.csv', header=None))
preds=dt[:,2]
preds_new=np.reshape(preds, (-1,54))

df = pd.DataFrame(preds_new, columns=transform_list)  

df.to_csv(output_path+'similarity_matrix.csv', index=False)

In [ ]:
from pytz import timezone 
from datetime import datetime
cur_time=datetime.now(timezone("Asia/Kolkata")).strftime('%d-%m-%Y-%H-%M-%S')
name = cur_time + '_' + end 
os.rename(tempfolder, name)
source = '{}'.format(name)+'/'
dest=shutil.move(source, destination)

In [ ]:
print(dest)